- I do a print("Hello") so I make sure Im connected. Seems to lose connection once in a while.

In [59]:
print("Hello World")

Hello World


1. Create a Jupyter Notebook using Agent Platform Colab Enterprise.
2. Create a Python function that uses Gemini to classify user questions into one of the following
categories: Employment, General Information, Emergency Services, or Tax Related
3. Create a second function that generates social media posts for government announcements like
weather emergencies, holidays, school closings, etc.
4. Write unit tests for each function using pytest.
5. Use the Google Evaluation API to evaluate and compare Gemini responses from different
prompts.
6. Upload the Notebook to GitHub for grading.

Functions that return results from LLMs need to be tested
like any other function
- Functions that return deterministic results are simple
	- Using an LLM for classification or sentiment analysis are examples
	- Tested like any other function
	- The response from the model is either right or wrong
- Model responses that are indeterminate are more difficult to test
	- There is no single right answer to: “Write me a response to this email”
	- To automate testing use an LLM to evaluate correctness

In [60]:
!gcloud config get-value project

qwiklabs-gcp-02-657b98113afe


Get PROJECT_ID

In [61]:
from google import genai
from google.genai.types import HttpOptions

PROJECT_ID = "qwiklabs-gcp-02-657b98113afe"
LOCATION = "us-central1"
MODEL = "gemini-2.5-flash"

genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1")
)

In [62]:
CATEGORIES = [
    "Employment",
    "General Information",
    "Emergency Services",
    "Tax Related"
]

# Business Use Case
Classify public sector questions into one of four categories:
 - Employment
 - General Information
 - Emergency Services
 - Tax Related

In [63]:
print(CATEGORIES)
## Testing

['Employment', 'General Information', 'Emergency Services', 'Tax Related']


In [64]:
def classify_question_basic(question: str) -> str:
    prompt = f"""
Classify the following question into one category:

Categories:
- Employment
- General Information
- Emergency Services
- Tax Related

Question:
{question}

Return only the category name.
"""

    response = genai_client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text.strip()

- Basic classifier to put questions into one of the defined categories.

In [65]:
question = "How do I apply for a city job?"

result = classify_question_basic(question)

print(f"Question: {question}")
print(f"Category: {result}")

## Testing

Question: How do I apply for a city job?
Category: Employment


In [66]:
question = "How do I pay my taxes?"

result = classify_question_basic(question)

print(f"Question: {question}")
print(f"Category: {result}")

## Testing

Question: How do I pay my taxes?
Category: Tax Related


In [67]:
def classify_question_enterprise(question: str) -> str:
    prompt = f"""
You are a public sector service request classifier.

Classify the user's question into exactly one of these categories:

- Employment: jobs, job applications, hiring, unemployment, workplace benefits, or career services
- General Information: office hours, locations, permits, public services, contacts, or general government questions
- Emergency Services: police, fire, medical emergency, urgent safety issues, disasters, or immediate danger
- Tax Related: property tax, income tax, tax payments, tax forms, refunds, or assessments

Rules:
- Return only one category name.
- Do not explain your answer.
- Do not return JSON.
- If the question is unclear or does not fit another category, return General Information.

Question:
{question}
"""

    response = genai_client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text.strip()

- Enterprise Classifier that uses clearer business definitions. Should give me a stricter output.

In [68]:
question = "Where do I pay my property taxes?"

result = classify_question_enterprise(question)

print(f"Question: {question}")
print(f"Category: {result}")

## Testing

Question: Where do I pay my property taxes?
Category: Tax Related


In [69]:
def generate_government_post(announcement: str) -> str:

    prompt = f"""
Create a professional government social media post.

Requirements:
- Clear and concise
- Appropriate for the general public
- Professional tone
- Include only details provided in the announcement
- Do not invent city names, dates, times, or locations
- Do not include placeholders such as [Your City] or [Insert Date]
- Keep under 100 words

Announcement:
{announcement}
"""

    response = genai_client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text.strip()

- Announcement Generator - Generate social media posts for public sector
announcements such as weather, holidays, road closures and school closings.

In [70]:
announcement = """
Due to severe winter weather advisory, all city offices will be closed on January 3rd.
Essential services will remain available.
"""

post = generate_government_post(announcement)

print(post)

## Testing

Due to a severe winter weather advisory, all city offices will be closed on January 3rd. Essential services will remain available.


In [71]:
announcement = """
Due to severe Wind Advisary, You will need to batten down the hatches.
Essential services will remain available.
"""

post = generate_government_post(announcement)

print(post)

## Testing

**Public Safety Announcement:**

A severe Wind Advisory is in effect. Residents are advised to secure loose outdoor items to ensure safety. Essential services will remain available. Your cooperation helps keep our community safe.


In [72]:
announcement = """
A Semi-Truck rolled over and spilled Legos everywhere. Be careful and wear shoes.
"""

post = generate_government_post(announcement)

print(post)

## Testing

Public Safety Announcement:

A semi-truck incident has resulted in debris on the roadway and surrounding areas. For your safety, please exercise extreme caution if traveling through the affected vicinity. We advise wearing appropriate footwear to prevent injury. Thank you for your cooperation.


- Installed ipytest to work with Jupyter notebooks.

In [73]:
!pip install -q ipytest

In [74]:
import ipytest

ipytest.autoconfig()

In [75]:
%%ipytest -vv -s

def test_basic_classifier_returns_valid_category():
    result = classify_question_basic(
        "How do I apply for a government job?"
    )

    print("\nBasic Classifier Result:", result)

    assert result in CATEGORIES


def test_basic_classifier_tax_related():
    result = classify_question_basic(
        "Where do I pay my property taxes?"
    )

    print("\nBasic Tax Result:", result)

    assert result == "Tax Related"


def test_enterprise_classifier_returns_valid_category():
    result = classify_question_enterprise(
        "How do I apply for a city maintenance job?"
    )

    print("\nEnterprise Classifier Result:", result)

    assert result in CATEGORIES


def test_enterprise_classifier_tax_related():
    result = classify_question_enterprise(
        "Where do I pay my property taxes?"
    )

    print("\nEnterprise Tax Result:", result)

    assert result == "Tax Related"


def test_generate_government_post_returns_text():
    announcement = """
    City of Aurora Bay offices will be closed Monday, June 15th due to severe wind advisory.
    Emergency services will remain available.
    """

    result = generate_government_post(announcement)

    print("\nGovernment Post:")
    print(result)

    assert isinstance(result, str)
    assert len(result.strip()) > 0

======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collecting ... collected 5 items

t_ae43f7b874314395b134dabbe60184ea.py::test_basic_classifier_returns_valid_category 
Basic Classifier Result: Employment
PASSED
t_ae43f7b874314395b134dabbe60184ea.py::test_basic_classifier_tax_related 
Basic Tax Result: Tax Related
PASSED
t_ae43f7b874314395b134dabbe60184ea.py::test_enterprise_classifier_returns_valid_category 
Enterprise Classifier Result: Employment
PASSED
t_ae43f7b874314395b134dabbe60184ea.py::test_enterprise_classifier_tax_related 
Enterprise Tax Result: Tax Related
PASSED
t_ae43f7b874314395b134dabbe60184ea.py::test_generate_government_post_returns_text 
Government Post:
**City of Aurora Bay Public Service Announcement**

City of Aurora Bay offices will be c

- When I looked up these warnings, sources said to ignore, they are common.

- Additional testing comparing my basic vs more enterprise versions.

In [77]:
print(classify_question_basic(
    "I lost my job and need help paying my taxes."
))

print(classify_question_enterprise(
    "I lost my job and need help paying my taxes."
))

# Additional Testing

Tax Related
Tax Related


In [78]:
print(classify_question_basic(
    "My business is closed because of flooding."
))

print(classify_question_enterprise(
    "My business is closed because of flooding."
))

# Additional Testing

Emergency Services
General Information


# Step 5 - Google Evaluation API

In [83]:
!pip install -q google-cloud-aiplatform

In [84]:
import pandas as pd
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate
import vertexai

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

In [85]:
eval_dataset = pd.DataFrame({
    "question": [
        "How do I apply for a government job?",
        "Where do I pay my property taxes?",
        "There is a fire near my house. Who do I call?",
        "What time does city hall open?",
        "I lost my job and need help paying taxes."
    ],
    "expected_category": [
        "Employment",
        "Tax Related",
        "Emergency Services",
        "General Information",
        "Tax Related"
    ]
})

In [86]:
eval_dataset["basic_response"] = eval_dataset["question"].apply(
    classify_question_basic
)

eval_dataset["enterprise_response"] = eval_dataset["question"].apply(
    classify_question_enterprise
)

eval_dataset

,question,expected_category,basic_response,enterprise_response
0,How do I apply for a government job?,Employment,Employment,Employment
1,Where do I pay my property taxes?,Tax Related,Tax Related,Tax Related
2,There is a fire near my house. Who do I call?,Emergency Services,Emergency Services,Emergency Services
3,What time does city hall open?,General Information,General Information,General Information
4,I lost my job and need help paying taxes.,Tax Related,Tax Related,Tax Related


In [87]:
classification_metric = PointwiseMetric(
    metric="classification_accuracy",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "accuracy": (
                "Does the response match the expected category? "
                "The response should be exactly one of: Employment, General Information, "
                "Emergency Services, or Tax Related."
            )
        },
        rating_rubric={
            "1": "The response does not match the expected category.",
            "5": "The response exactly matches the expected category."
        },
        input_variables=["question", "expected_category", "response"]
    )
)

Evaluating my Basic version.

In [88]:
basic_eval_data = eval_dataset.rename(
    columns={"basic_response": "response"}
)[["question", "expected_category", "response"]]

basic_eval_task = EvalTask(
    dataset=basic_eval_data,
    metrics=[classification_metric],
    experiment="basic-classifier-eval"
)

basic_result = basic_eval_task.evaluate()

basic_result.summary_metrics

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 5 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 5/5 [00:09<00:00,  1.84s/it]
INFO:vertexai.evaluation._evaluation:All 5 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:9.226223192999896 seconds


{'row_count': 5,
 'classification_accuracy/mean': np.float64(5.0),
 'classification_accuracy/std': 0.0}

Evaluating the more Enterprise version.

In [89]:
enterprise_eval_data = eval_dataset.rename(
    columns={"enterprise_response": "response"}
)[["question", "expected_category", "response"]]

enterprise_eval_task = EvalTask(
    dataset=enterprise_eval_data,
    metrics=[classification_metric],
    experiment="enterprise-classifier-eval"
)

enterprise_result = enterprise_eval_task.evaluate()

enterprise_result.summary_metrics

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 5 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 5/5 [00:17<00:00,  3.54s/it]
INFO:vertexai.evaluation._evaluation:All 5 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:17.724809757999537 seconds


{'row_count': 5,
 'classification_accuracy/mean': np.float64(5.0),
 'classification_accuracy/std': 0.0}

Gotta Compare them.

In [91]:
print("Basic classifier results:")
print(basic_result.summary_metrics)

print("\nEnterprise classifier results:")
print(enterprise_result.summary_metrics)

Basic classifier results:
{'row_count': 5, 'classification_accuracy/mean': np.float64(5.0), 'classification_accuracy/std': 0.0}

Enterprise classifier results:
{'row_count': 5, 'classification_accuracy/mean': np.float64(5.0), 'classification_accuracy/std': 0.0}
